# HadCRUT5: quickstart

Download the HadCRUT5 infilled ensemble-mean NetCDF and plot both the
global-mean temperature anomaly time series and the latest monthly map.

See [`docs/hadcrut5/README.md`](../docs/hadcrut5/README.md) for the full
reference. No authentication required; `pip install requests xarray netcdf4
matplotlib cartopy numpy`.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
SUB_VERSION = "5.0.2.0"
PRODUCT = "ensemble_mean.infilled"  # or "ensemble_mean" for non-infilled
OUTPUT_DIR = "../data/hadcrut5"
OUTPUT_FILENAME = "hadcrut5_ensemble_mean_infilled.nc"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests

print(f"Python       {sys.version.split()[0]}")
for pkg in ["requests", "xarray", "matplotlib", "cartopy", "netcdf4", "numpy"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download the ensemble-mean NetCDF

A single HTTP GET. The infilled ensemble-mean file is a few tens of MB.

In [ ]:
from scripts.hadcrut5_download import download  # noqa: E402

output_path = download(
    sub_version=SUB_VERSION,
    product=PRODUCT,
    output_dir=OUTPUT_DIR,
    output_filename=OUTPUT_FILENAME,
)
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

HadCRUT5 NetCDF files have `time`, `latitude`, `longitude` dimensions and
the main variable `tas_mean` (near-surface temperature anomaly in K).

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

## Global-mean time series

Global-mean temperature anomalies from 1850. Use cos-latitude weighting to
get a true area-weighted mean.

In [ ]:
var = "tas_mean" if "tas_mean" in ds.data_vars else list(ds.data_vars)[0]
anom = ds[var]

weights = np.cos(np.deg2rad(ds["latitude"]))
gmst = anom.weighted(weights).mean(("latitude", "longitude"))
annual = gmst.resample(time="1YE").mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(gmst["time"], gmst, color="lightgray", linewidth=0.6, label="Monthly")
ax.plot(annual["time"], annual, color="C3", linewidth=1.5, label="Annual mean")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Year")
ax.set_ylabel("Global-mean temperature anomaly (K)\nrelative to 1961-1990")
ax.set_title("HadCRUT5 global-mean temperature anomaly, 1850 to present")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Latest monthly map

The most recent month's spatial pattern of anomalies. The 5 degree grid
means each cell is roughly 500 km on a side; HadCRUT5 is a climate-scale
product, not a regional one.

In [ ]:
latest = anom.isel(time=-1)
latest_date = str(anom["time"].values[-1])[:7]

fig = plt.figure(figsize=(11, 5))
ax = plt.axes(projection=ccrs.Robinson())
latest.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=-3, vmax=3,
    cbar_kwargs={"label": "Temperature anomaly (K) vs 1961-1990"},
)
ax.coastlines(resolution="110m", linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax.set_title(f"HadCRUT5 monthly anomaly, {latest_date}")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/hadcrut5/README.md`](../docs/hadcrut5/README.md)
- **Ensemble uncertainty**: pull individual ensemble members (set `MEMBER`
  in the config to an integer 1-200) to quantify observational uncertainty
  on your derived quantities.
- **Rebaseline**: HadCRUT5 uses 1961-1990. To compare with datasets on a
  different baseline (NASA GISS 1951-1980, ERA5 1991-2020), subtract the
  mean over the appropriate years.
- **Compare with HadCET**: HadCET covers central England as a single time
  series from 1659. HadCRUT5 is global from 1850. Overlaying them shows
  how a single-region record relates to the global pattern.